In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [5]:
matmul=r"""

#include <cstdio>
#include <cuda_runtime.h>

__global__ void matmul(const float* A, const float* B, float* C, int M, int N, int K){
    
    int col = blockIdx.x*blockDim.x+threadIdx.x;
    int row = blockIdx.y*blockDim.y+threadIdx.y;

    if (row<M && col<N){
        float sum=0.0f;

        for(int k=0; k<K; k++){
            float a=A[row*K+K];
            float b=B[k*N+col];
            sum+=a*b;
        }
        C[row*N+col]=sum;
    }
}

int main(){
    const int M=100, K=200, N=300;
    
    float *A, *B, *C;
    cudaMallocManaged(&A, M*K*sizeof(float));
    cudaMallocManaged(&B, K*N*sizeof(float));
    cudaMallocManaged(&C, M*N*sizeof(float));

    for(int i=0; i<M*K; i++) A[i]=1.0f;
    for(int i=0; i<K*N; i++) B[i]=2.0f;

    dim3 block(16, 16);
    dim3 grid((N+block.x-1)/block.x, (M+block.y-1)/block.y);
    
    matmul<<<block, grid>>>(A, B, C, M, N, K);
    cudaDeviceSynchronize();

    printf("C[0][0] = %f\n", C[0]);
    printf("C[%d][%d] = %f\n", M-1, N-1, C[(M - 1) * N + (N - 1)]);

    cudaFree(A);
    cudaFree(B);
    cudaFree(C);

    return 0;
}

"""

with open('matmul.cu', 'w') as f:
    f.write(matmul)

In [6]:
!nvcc -arch=sm_75 matmul.cu -o matmul

In [7]:
!./matmul

C[0][0] = 400.000000
C[99][299] = 0.000000
